## Cleaning and Standardising the Silver Layer

This notebook walks through a complete data cleaning pipeline for the deduplicated **pupils** and **schools** datasets, producing analysis-ready tables in the silver layer.

> **Note:** All data in this module is **entirely synthetic** and does not represent any real schools, pupils, or individuals.

### What we'll do

| Step | Purpose |
| --- | --- |
| **Profile** | Discover inconsistencies — casing, abbreviations, nulls, sentinel values |
| **Standardise labels** | Map variants to the *most informative* canonical form (e.g. `'Male'` over `'M'`, `'Year 8'` over `'Yr 8'`) |
| **Extract JSON fields** | Pull structured data out of `metadata_json` into proper SQL columns |
| **Cast types correctly** | Ensure dates are `DATE`, booleans are `BOOLEAN`, percentages are `DOUBLE` |
| **Handle missing data** | Categorical fields → `'Unknown'`; non-categorical fields (dates, numbers, addresses) → `NULL` |
| **Separate contact details** | Extract parent/guardian contact information (including from audit duplicate rows) into a dedicated table |

### Source and output tables

| Source table | Rows | Description |
| --- | --- | --- |
| `silver.pupils_deduplicated` | 111 | Winning rows from pupil deduplication |
| `silver.schools_deduplicated` | 75 | Winning rows from school deduplication |
| `silver.audit_pupils_duplicate_rows` | 17 | Discarded pupil duplicates (may contain unique contact details) |
| `silver.audit_schools_duplicate_rows` | 1 | Discarded school duplicate |

| Output table | Purpose |
| --- | --- |
| `silver.pupils_cleaned` | Standardised pupils with correct types and missing data handled |
| `silver.schools_cleaned` | Standardised schools with correct types and missing data handled |
| `silver.contacts_cleaned` | Parent/guardian contact details extracted from all sources |

## Part 1: Cleaning the Schools Data

The schools table has casing differences, abbreviations, nulls, and **sentinel values** (placeholder strings like `'N/A'` or `'Unknown'` that represent missing data). Identifying and normalising these is a key part of cleaning.

We start by profiling every string column to surface the issues, then build lookup tables to recode them.

In [0]:
USE CATALOG catalog_40_copper_analyst_training;

In [0]:
-- Profile all string columns in the schools table
-- Surfaces casing differences, abbreviations, nulls, empty strings, and sentinel values

SELECT 'school_type' AS column_name, school_type AS raw_value, COUNT(*) AS row_count
FROM silver.schools_deduplicated GROUP BY school_type
UNION ALL
SELECT 'status', status, COUNT(*)
FROM silver.schools_deduplicated GROUP BY status
UNION ALL
SELECT 'phase', phase, COUNT(*)
FROM silver.schools_deduplicated GROUP BY phase
UNION ALL
SELECT 'region', region, COUNT(*)
FROM silver.schools_deduplicated GROUP BY region
UNION ALL
SELECT 'ofsted_rating', metadata_json:ofsted_rating, COUNT(*)
FROM silver.schools_deduplicated GROUP BY metadata_json:ofsted_rating
ORDER BY column_name, raw_value

### Standardising school fields

The fields above have the following issues:

| Field | Issues | Approach |
| --- | --- | --- |
| **school_type** | `'academy'`, `'Acad'`, `'N/A'` | Lookup table; unmatched → `'Unknown'` |
| **status** | `'open'` (lowercase), `'Unknown'` | `CASE WHEN`; unmatched → `'Unknown'` |
| **phase** | `'primary'`, `'secondary'`, `'Sec'` | Lookup table; unmatched → `'Unknown'` |
| **location** | `'SHEFFIELD'`, `'liverpool'`, nulls | `INITCAP()` + `COALESCE(..., 'Unknown')` |
| **region** | `'east of england'`, `'north east'`, nulls | `INITCAP()` + `COALESCE(..., 'Unknown')` |

All of these are categorical fields — they appear as analysis dimensions in filters, group-bys, chart axes, and map visuals. Using `'Unknown'` for missing data keeps those rows visible in reports rather than silently dropping them.

### School type lookup

To recode the values we can create a temporary view to serve as a lookup table which we can join to the data when creating the finalised 'clean' schools table.

Unrecognised values (`'N/A'`, empty strings, nulls) won't match any entry and become `'Unknown'` via `COALESCE(lookup.canonical_value, 'Unknown')`.

In [0]:
-- School type: casing variants, abbreviations, and 'N/A' sentinel values
CREATE OR REPLACE TEMP VIEW school_type_lookup AS
SELECT * FROM VALUES
  ('academy',     'Academy')
  ,('acad',       'Academy')
  ,('maintained', 'Maintained')
  ,('free school', 'Free School')
AS t(raw_value, canonical_value);

SELECT * FROM school_type_lookup;

### Phase lookup

Phase follows the same lookup pattern as school type. It has casing variants (`'primary'`, `'secondary'`), an abbreviation (`'Sec'` → `'Secondary'`), nulls, and empty strings.

In [0]:
-- Phase: casing variants and 'Sec' abbreviation
-- NULL and empty string won't match → become NULL via LEFT JOIN

CREATE OR REPLACE TEMP VIEW phase_lookup AS
SELECT * FROM VALUES
  ('primary',      'Primary')
  ,('secondary',   'Secondary')
  ,('sec',         'Secondary')
  ,('all-through', 'All-through')
AS t(raw_value, canonical_value);

SELECT * FROM phase_lookup;

### Enriching from discarded duplicates

During deduplication, the winning row was chosen as the *most complete* record. But the discarded duplicates in `audit_schools_duplicate_rows` may contain field values that the winner lacks — for example, an Ofsted rating or region that only appeared on the losing row.

To avoid losing data, we:

1. **Pre-aggregate** the audit rows into a single "best available" row per school/term/year, taking the `MAX` of each field (which picks the non-null value when one exists)
2. **`COALESCE`** each field in the final query: prefer the winning value, fall back to the audit value

This is a **defensive pattern** — it protects against data loss without changing the cardinality (same number of rows).

In [0]:
-- Pre-aggregate discarded school duplicates into best available values per key
-- MAX picks a non-null value when one exists alongside nulls

CREATE OR REPLACE TEMP VIEW audit_schools_enrichment AS

SELECT school_natural_key
      ,MAX(location)                           AS location
      ,MAX(school_type)                        AS school_type
      ,MAX(status)                             AS status
      ,MAX(phase)                              AS phase
      ,MAX(region)                             AS region
      ,MAX(metadata_json:ofsted_rating)        AS ofsted_rating
      ,MAX(metadata_json:last_inspection)       AS last_inspection
      ,MAX(metadata_json:pupil_premium_pct)    AS pupil_premium_pct
FROM silver.audit_schools_duplicate_rows
GROUP BY school_natural_key;

SELECT * FROM audit_schools_enrichment;

### Bringing it all together: `schools_cleaned`

The final schools table applies all standardisations:

| Transformation | Technique | Fields affected |
| --- | --- | --- |
| **School name** | URN-based lookup (longest, properly-cased name) | `school_name` |
| **School type** | Lookup table + `COALESCE(..., 'Unknown')` | `school_type` |
| **Phase** | Lookup table + `COALESCE(..., 'Unknown')` | `phase` |
| **Status** | `CASE WHEN` with `ELSE 'Unknown'` | `status` |
| **Location / Region** | `INITCAP()` + `COALESCE(..., 'Unknown')` | `location`, `region` |
| **Ofsted rating** | Extracted from JSON + `COALESCE(..., 'Unknown')` | `ofsted_rating` |
| **Type casting** | `TO_DATE()` / `CAST()` | `last_inspection` → `DATE`, `pupil_premium_pct` → `DOUBLE` |
| **Audit enrichment** | `LEFT JOIN` audit + `COALESCE(winner, audit)` | All fields — fills gaps from discarded duplicates |

> **Note:** In this dataset, the deduplication process already selected the most complete record as the winner — no fields were actually recovered from the audit table. However, this enrichment step is **best practice** because you cannot always guarantee the winning row is the most complete. Including it as a defensive pattern ensures no data is silently lost, at zero cost when there are no gaps to fill.

In [0]:
-- Write the fully cleaned schools table
-- COALESCE prefers the winning value, falls back to the audit value for each field
-- Categorical fields then get COALESCE(..., 'Unknown'); non-categoricals stay NULL

CREATE OR REPLACE TEMPORARY VIEW schools_cleaned AS

SELECT s.school_urn
      ,s.school_name                                     AS school_name
      ,COALESCE(
        NULLIF(INITCAP(TRIM(COALESCE(s.location, ''))), ''),
        NULLIF(INITCAP(TRIM(COALESCE(ase.location, ''))), ''),
        'Unknown'
      )                                                               AS location
      ,COALESCE(
        stl.canonical_value,
        stl_ase.canonical_value,
        'Unknown'
      )                                                               AS school_type
      ,CASE
        WHEN LOWER(TRIM(COALESCE(s.status, ase.status, ''))) = 'open'   THEN 'Open'
        WHEN LOWER(TRIM(COALESCE(s.status, ase.status, ''))) = 'closed' THEN 'Closed'
        ELSE 'Unknown'
      END                                                             AS status
      ,COALESCE(pl.canonical_value, pl_ase.canonical_value, 'Unknown') AS phase
      ,COALESCE(
        NULLIF(INITCAP(TRIM(COALESCE(s.region, ''))), ''),
        NULLIF(INITCAP(TRIM(COALESCE(ase.region, ''))), ''),
        'Unknown'
      )                                                               AS region
      ,COALESCE(
        NULLIF(REPLACE(s.metadata_json:ofsted_rating, '"', ''), ''),
        NULLIF(REPLACE(ase.ofsted_rating, '"', ''), ''),
        'Unknown'
      )                                                               AS ofsted_rating
      ,COALESCE(
        TO_DATE(REPLACE(s.metadata_json:last_inspection, '"', ''), 'yyyy-MM-dd'),
        TO_DATE(REPLACE(ase.last_inspection, '"', ''), 'yyyy-MM-dd')
      )                                                               AS last_inspection
      ,COALESCE(
        CAST(s.metadata_json:pupil_premium_pct AS DOUBLE),
        CAST(ase.pupil_premium_pct AS DOUBLE)
      )                                                               AS pupil_premium_pct
      ,s.term
      ,s.year
FROM silver.schools_deduplicated s
LEFT JOIN audit_schools_enrichment ase
  ON s.school_natural_key = ase.school_natural_key
LEFT JOIN school_type_lookup stl
  ON LOWER(TRIM(COALESCE(s.school_type, ''))) = stl.raw_value
LEFT JOIN school_type_lookup stl_ase
  ON LOWER(TRIM(COALESCE(ase.school_type, ''))) = stl_ase.raw_value
LEFT JOIN phase_lookup pl
  ON LOWER(TRIM(COALESCE(s.phase, ''))) = pl.raw_value
LEFT JOIN phase_lookup pl_ase
  ON LOWER(TRIM(COALESCE(ase.phase, ''))) = pl_ase.raw_value;

SELECT * FROM schools_cleaned LIMIT 12;

In [0]:
SELECT * FROM silver.schools_deduplicated

### Validating `schools_cleaned`

After writing the table, we verify that:

* The **row count matches the source** (75 rows, 21 distinct schools)
* All categorical fields (`school_type`, `status`, `phase`, `location`, `region`, `ofsted_rating`) have **zero NULLs** — missing values are now `'Unknown'`
* Only non-categorical fields (`last_inspection`, `pupil_premium_pct`) may still have NULLs where data is genuinely absent

### Validating `schools_cleaned`

After writing the table, we check that:

* The **row count matches the source** (75 rows)
* Null counts are as expected — e.g. `region` is mostly null because it was only populated in the summer 2025 source
* No data was accidentally dropped by the joins

In [0]:
-- Validate: row count matches source (75), categorical fields have no NULLs
-- Only dates and numbers may still have NULLs

SELECT COUNT(*)               AS total_rows
      ,COUNT(DISTINCT school_urn) AS distinct_schools
      ,SUM(CASE WHEN school_type = 'Unknown' THEN 1 ELSE 0 END)    AS unknown_school_type
      ,SUM(CASE WHEN status = 'Unknown' THEN 1 ELSE 0 END)         AS unknown_status
      ,SUM(CASE WHEN phase = 'Unknown' THEN 1 ELSE 0 END)          AS unknown_phase
      ,SUM(CASE WHEN location = 'Unknown' THEN 1 ELSE 0 END)       AS unknown_location
      ,SUM(CASE WHEN region = 'Unknown' THEN 1 ELSE 0 END)         AS unknown_region
      ,SUM(CASE WHEN ofsted_rating = 'Unknown' THEN 1 ELSE 0 END)  AS unknown_ofsted
      ,COUNT(last_inspection)  AS non_null_inspection
      ,COUNT(pupil_premium_pct) AS non_null_pp_pct
FROM schools_cleaned;

---

## Part 2: Cleaning the Pupils Data

The pupils table has many of the same issues as schools — casing differences, abbreviations, sentinel values, and nulls — but also introduces some new challenges:

* **Mixed date formats** — `date_of_birth` is stored as a string with three different formats (`'2012-03-15'`, `'15/02/2011'`, `'03-22-2013'`) that need parsing into a proper `DATE` type
* **Rich JSON metadata** — `metadata_json` contains pupil attributes (`sen_status`, `fsm_eligible`, `attendance_pct`), address details, and parent/guardian contact information that all need extracting into flat columns
* **Name casing errors** — data entry inconsistencies like `katie JONES` and `LIAM garcia` need standardising with `INITCAP()`

We follow the same approach as Part 1: profile first, then standardise labels with lookup tables, extract and cast JSON fields, and recode missing categorical data to `'Unknown'`.

We'll start by profiling the key string columns to see exactly what needs fixing.

In [0]:
-- Profile key string columns in the pupils table to surface inconsistencies
-- Look for: casing differences, abbreviations, nulls, empty strings, sentinel values

SELECT 'gender' AS column_name, gender AS raw_value, COUNT(*) AS row_count
FROM silver.pupils_deduplicated GROUP BY gender
UNION ALL
SELECT 'year_group', year_group, COUNT(*)
FROM silver.pupils_deduplicated GROUP BY year_group
UNION ALL
SELECT 'ethnicity', ethnicity, COUNT(*)
FROM silver.pupils_deduplicated GROUP BY ethnicity
UNION ALL
SELECT 'sen_status', metadata_json:sen_status, COUNT(*)
FROM silver.pupils_deduplicated GROUP BY metadata_json:sen_status
ORDER BY column_name, raw_value

### Exploring the `metadata_json` structure

Before extracting fields from JSON, we need to understand what's inside. The query below uses Databricks' `:` path syntax to pull out each nested key.

Pupils' `metadata_json` contains:
* **Pupil attributes** — `sen_status`, `fsm_eligible`, `attendance_pct`
* **Address** — `address.line1`, `address.city`, `address.postcode`
* **Contact details** — `contact.parent_name`, `contact.phone`, `contact.email`, `contact.address.*`

In [0]:
-- Examine the JSON structure to identify fields to extract
-- Pupils metadata_json contains: address, contact, sen_status, fsm_eligible, attendance_pct

SELECT pupil_id
      ,metadata_json:sen_status         AS sen_status
      ,metadata_json:fsm_eligible       AS fsm_eligible
      ,metadata_json:attendance_pct     AS attendance_pct
      ,metadata_json:address.line1      AS address_line1
      ,metadata_json:address.city       AS address_city
      ,metadata_json:address.postcode   AS address_postcode
      ,metadata_json:contact.parent_name AS contact_parent_name
      ,metadata_json:contact.phone      AS contact_phone
      ,metadata_json:contact.email      AS contact_email
FROM silver.pupils_deduplicated
WHERE metadata_json:contact IS NOT NULL
ORDER BY pupil_id
LIMIT 10

### Standardising gender with a lookup table

Gender has five variants in the source data: `NULL`, `'F'`, `'Female'`, `'M'`, `'Male'`. We standardise to the **most informative** labels — `'Male'` and `'Female'` — using a lookup table joined on `LOWER(TRIM(...))`. Missing gender is recoded to `'Unknown'` via `COALESCE(lookup.canonical_value, 'Unknown')` in the final table.

In [0]:
-- Gender has 5 variants: NULL, F, Female, M, Male
-- Standardise to the MOST INFORMATIVE label: 'Male' and 'Female'
-- NULL gender remains NULL (genuine missing data)

CREATE OR REPLACE TEMP VIEW gender_lookup AS
SELECT * FROM VALUES
  ('f',      'Female')
  ,('female', 'Female')
  ,('m',      'Male')
  ,('male',   'Male')
AS t(raw_value, canonical_value);

SELECT * FROM gender_lookup;

### Year group and ethnicity lookups

**Year group** has abbreviations (`Yr 8` → `Year 8`), casing variants (`year 6`), nulls, and empty strings. We use a lookup table because the set of valid values is well-defined.

**Ethnicity** has mixed casing (`asian british` → `Asian British`), sentinel values (`N/A`), and 82 null rows (ethnicity was only populated from summer 2025 onwards). Again, a lookup table is the best approach.

For both fields, unmatched values (nulls, empty strings, sentinel values) will become `'Unknown'` in the cleaned table via `COALESCE(lookup.canonical_value, 'Unknown')`. This ensures categorical fields always have an explicit value, making them safe to use in `GROUP BY`, `COUNT`, chart axes, and cross-tabulations without accidentally dropping rows.

In [0]:
-- Year group: abbreviations and casing → standardise to 'Year N' format
-- The lookup key is LOWER(TRIM(...)) so 'Yr 8', 'yr 8', and ' Year 8 ' all match

CREATE OR REPLACE TEMP VIEW year_group_lookup AS
SELECT * FROM VALUES
  ('year 6',  'Year 6')
  ,('year 7', 'Year 7')
  ,('year 8', 'Year 8')
  ,('year 9', 'Year 9')
  ,('yr 6',   'Year 6')
  ,('yr 7',   'Year 7')
  ,('yr 8',   'Year 8')
  ,('yr 9',   'Year 9')
AS t(raw_value, canonical_value);

SELECT * FROM year_group_lookup;

### Ethnicity lookup

Ethnicity has mixed casing (`'asian british'` → `'Asian British'`), a sentinel value (`'N/A'`), and 82 null rows (ethnicity was only populated from summer 2025 onwards). The lookup maps recognised values to Title Case canonical forms. 

In [0]:
-- Ethnicity: mixed casing and N/A → standardise to Title Case
-- 'N/A', empty strings, and NULLs won't match any entry → become NULL via the LEFT JOIN

CREATE OR REPLACE TEMP VIEW ethnicity_lookup AS
SELECT * FROM VALUES
  ('white british',      'White British')
  ,('white irish',       'White Irish')
  ,('other white',       'Other White')
  ,('mixed',             'Mixed')
  ,('mixed white asian', 'Mixed White Asian')
  ,('mixed white black', 'Mixed White Black')
  ,('asian',             'Asian')
  ,('asian indian',      'Asian Indian')
  ,('asian british',     'Asian British')
  ,('black african',     'Black African')
  ,('other',             'Other')
AS t(raw_value, canonical_value);

SELECT * FROM ethnicity_lookup;

### Name casing with `INITCAP()`

Some pupil names have inconsistent casing from data entry errors — `katie JONES`, `LIAM garcia`. `INITCAP()` converts each word to Title Case, which is the standard for personal names.

> **Caution:** `INITCAP()` works well for most English names but can produce incorrect results for names with particles (e.g. `McDonald` → `Mcdonald`). For this synthetic dataset it's sufficient; production systems may need a more sophisticated approach.

In [0]:
-- Show rows where name casing needs fixing
-- INITCAP() converts each word to Title Case: 'katie' → 'Katie', 'JONES' → 'Jones'

SELECT pupil_id
      ,first_name       AS first_name_raw
      ,INITCAP(first_name) AS first_name_clean
      ,last_name        AS last_name_raw
      ,INITCAP(last_name)  AS last_name_clean
FROM silver.pupils_deduplicated
WHERE first_name != INITCAP(first_name)
   OR last_name  != INITCAP(last_name)
ORDER BY pupil_id

### SEN status (from `metadata_json`)

SEN status is stored inside `metadata_json` and extracted with `:` path syntax. It has casing issues (`None` vs `none`), sentinel values (`N/A`, `Unknown`), and nulls.

We standardise to three canonical values: `'EHCP'`, `'SEN Support'`, and `'None'` — with genuine unknowns (`N/A`, existing `'Unknown'`, `NULL`) all mapped to `'Unknown'`.

In [0]:
-- SEN status: standardise with CASE WHEN
-- 'None' and 'none' both mean 'no SEN provision' → 'None'
-- 'N/A', 'Unknown', and NULL all mean genuinely unknown → 'Unknown'

SELECT metadata_json:sen_status AS sen_status_raw
      ,CASE
        WHEN LOWER(TRIM(metadata_json:sen_status)) = 'ehcp'        THEN 'EHCP'
        WHEN LOWER(TRIM(metadata_json:sen_status)) = 'sen support' THEN 'SEN Support'
        WHEN LOWER(TRIM(metadata_json:sen_status)) IN ('none')     THEN 'None'
        ELSE 'Unknown'
      END AS sen_status_clean
      ,COUNT(*) AS cnt
FROM silver.pupils_deduplicated
GROUP BY metadata_json:sen_status
ORDER BY sen_status_clean, sen_status_raw

#### Handling mixed date formats

The `date_of_birth` field has three different formats: `'2012-03-15'` (ISO), `'15/02/2011'` (UK), and `'03-22-2013'` (US). Rather than failing on the first non-matching format, we use:

```sql
COALESCE(
  TRY_TO_DATE(date_of_birth, 'yyyy-MM-dd'),
  TRY_TO_DATE(date_of_birth, 'dd/MM/yyyy'),
  TRY_TO_DATE(date_of_birth, 'MM-dd-yyyy')
)
```

`TRY_TO_DATE` returns `NULL` instead of erroring on a bad parse. `COALESCE` then picks the first successful result.

#### Why `'Unknown'` for categorical fields?

Categorical fields (gender, year_group, ethnicity, sen_status) are analysis dimensions — they appear in `GROUP BY` clauses, chart axes, filters, and cross-tabulations. Using `'Unknown'` rather than `NULL` ensures:

* **SQL**: `GROUP BY gender` includes an explicit `'Unknown'` group instead of silently dropping nulls
* **Power BI**: Charts show an `'Unknown'` bar/slice rather than omitting missing data entirely
* **R / Python**: No need for special `na.rm` or `dropna()` handling — the category is always present
* **Counts**: `COUNT(gender)` counts all 111 rows, not just the 110 with known values

Non-categorical fields (dates, booleans, percentages, addresses) keep `NULL` because `'Unknown'` would conflict with their data types or be misleading (e.g. `'Unknown'` as a postcode).

In [0]:
-- Show how COALESCE(TRY_TO_DATE(...)) handles the three date formats found in date_of_birth
-- Each TRY_TO_DATE returns NULL if the format doesn't match, COALESCE picks the first success

SELECT date_of_birth AS raw_value
      ,TRY_TO_DATE(date_of_birth, 'yyyy-MM-dd') AS try_iso
      ,TRY_TO_DATE(date_of_birth, 'dd/MM/yyyy') AS try_uk
      ,TRY_TO_DATE(date_of_birth, 'MM-dd-yyyy') AS try_us
      ,COALESCE(
        TRY_TO_DATE(date_of_birth, 'yyyy-MM-dd'),
        TRY_TO_DATE(date_of_birth, 'dd/MM/yyyy'),
        TRY_TO_DATE(date_of_birth, 'MM-dd-yyyy')
      ) AS parsed_date
FROM silver.pupils_deduplicated
WHERE date_of_birth IN ('2012-03-15', '15/02/2011', '03-22-2013')
ORDER BY date_of_birth

### Enriching from discarded duplicates

As with schools, the discarded pupil duplicates in `audit_pupils_duplicate_rows` may contain metadata values that the winning row lacks. We apply the same defensive pattern:

1. **Pre-aggregate** audit rows per `pupil_id` / `term` / `year`
2. **`COALESCE`** each field in the final query: winning value first, audit value as fallback

This guarantees we never lose data that existed in the source, without changing the cardinality of the output table.

In [0]:
-- Pre-aggregate discarded pupil duplicates into best available values per key

CREATE OR REPLACE TEMP VIEW audit_pupils_enrichment AS

SELECT pupil_id
      ,term
      ,year
      ,MAX(gender)                             AS gender
      ,MAX(year_group)                         AS year_group
      ,MAX(ethnicity)                          AS ethnicity
      ,MAX(metadata_json:sen_status)           AS sen_status
      ,MAX(metadata_json:fsm_eligible)         AS fsm_eligible
      ,MAX(metadata_json:attendance_pct)       AS attendance_pct
      ,MAX(metadata_json:address.line1)        AS address_line1
      ,MAX(metadata_json:address.city)         AS address_city
      ,MAX(metadata_json:address.postcode)     AS address_postcode
FROM silver.audit_pupils_duplicate_rows
GROUP BY pupil_id, term, year;

SELECT * FROM audit_pupils_enrichment LIMIT 10;

#### Contact details

Contact information (`parent_name`, `phone`, `email`) is extracted separately into `silver.contacts_cleaned` — see the final section of this notebook.

### Bringing it all together: JSON extraction, type casting, and missing data handling

The final cleaned table applies all standardisations in a single `CREATE OR REPLACE TABLE` statement:

| Transformation | Technique | Fields affected |
| --- | --- | --- |
| **Label standardisation** | Lookup tables via `LEFT JOIN` | `gender`, `year_group`, `ethnicity` |
| **Name casing** | `INITCAP()` | `first_name`, `last_name` |
| **SEN status** | `CASE WHEN` with `ELSE 'Unknown'` | `sen_status` (from JSON) |
| **JSON extraction** | `:` path syntax | `fsm_eligible`, `attendance_pct`, `address_*` |
| **Type casting** | `CAST()` / `TRY_TO_DATE()` | `date_of_birth` → `DATE`, `fsm_eligible` → `BOOLEAN`, `attendance_pct` → `DOUBLE` |
| **Categorical missing data** | `COALESCE(..., 'Unknown')` | `gender`, `year_group`, `ethnicity`, `sen_status` |
| **Non-categorical missing data** | Stays as `NULL` | `date_of_birth`, `fsm_eligible`, `attendance_pct`, `address_*` |
| **Audit enrichment** | `LEFT JOIN` audit + `COALESCE(winner, audit)` | All fields — fills gaps from discarded duplicates |

> **Note:** In this dataset, the deduplication process already selected the most complete record as the winner — no fields were actually recovered from the audit table. However, this enrichment step is **best practice** because you cannot always guarantee the winning row is the most complete. Including it as a defensive pattern ensures no data is silently lost, at zero cost when there are no gaps to fill.

In [0]:
-- Preview all transformations + audit enrichment
-- COALESCE prefers winning value, falls back to audit value for each field

SELECT p.pupil_id
      ,INITCAP(p.first_name)                                        AS first_name
      ,INITCAP(p.last_name)                                         AS last_name
      ,COALESCE(gl.canonical_value, gl_ape.canonical_value, 'Unknown') AS gender
      ,COALESCE(
        TRY_TO_DATE(p.date_of_birth, 'yyyy-MM-dd'),
        TRY_TO_DATE(p.date_of_birth, 'dd/MM/yyyy'),
        TRY_TO_DATE(p.date_of_birth, 'MM-dd-yyyy')
      )                                                               AS date_of_birth
      ,p.school_urn
      ,COALESCE(ygl.canonical_value, ygl_ape.canonical_value, 'Unknown') AS year_group
      ,COALESCE(el.canonical_value, el_ape.canonical_value, 'Unknown')   AS ethnicity
      ,p.term
      ,p.year
      ,CASE
        WHEN LOWER(TRIM(COALESCE(p.metadata_json:sen_status, ape.sen_status))) = 'ehcp'        THEN 'EHCP'
        WHEN LOWER(TRIM(COALESCE(p.metadata_json:sen_status, ape.sen_status))) = 'sen support' THEN 'SEN Support'
        WHEN LOWER(TRIM(COALESCE(p.metadata_json:sen_status, ape.sen_status))) IN ('none')     THEN 'None'
        ELSE 'Unknown'
      END                                                             AS sen_status
      ,CAST(COALESCE(p.metadata_json:fsm_eligible, ape.fsm_eligible) AS BOOLEAN)   AS fsm_eligible
      ,CAST(COALESCE(p.metadata_json:attendance_pct, ape.attendance_pct) AS DOUBLE) AS attendance_pct
      ,COALESCE(REPLACE(p.metadata_json:address.line1, '"', ''), REPLACE(ape.address_line1, '"', ''))         AS address_line1
      ,COALESCE(REPLACE(p.metadata_json:address.city, '"', ''), REPLACE(ape.address_city, '"', ''))           AS address_city
      ,COALESCE(REPLACE(p.metadata_json:address.postcode, '"', ''), REPLACE(ape.address_postcode, '"', ''))   AS address_postcode
FROM silver.pupils_deduplicated p
LEFT JOIN audit_pupils_enrichment ape
  ON p.pupil_id = ape.pupil_id AND p.term = ape.term AND p.year = ape.year
LEFT JOIN gender_lookup gl
  ON LOWER(TRIM(p.gender)) = gl.raw_value
LEFT JOIN gender_lookup gl_ape
  ON LOWER(TRIM(ape.gender)) = gl_ape.raw_value
LEFT JOIN year_group_lookup ygl
  ON LOWER(TRIM(COALESCE(p.year_group, ''))) = ygl.raw_value
LEFT JOIN year_group_lookup ygl_ape
  ON LOWER(TRIM(COALESCE(ape.year_group, ''))) = ygl_ape.raw_value
LEFT JOIN ethnicity_lookup el
  ON LOWER(TRIM(COALESCE(p.ethnicity, ''))) = el.raw_value
LEFT JOIN ethnicity_lookup el_ape
  ON LOWER(TRIM(COALESCE(ape.ethnicity, ''))) = el_ape.raw_value
ORDER BY p.pupil_id, p.year, p.term
LIMIT 20

### Writing the cleaned pupils table

This cell applies **all** the standardisation steps above in a single `CREATE OR REPLACE TABLE` statement — lookups, `CASE WHEN`, `INITCAP()`, JSON extraction, type casting, and `COALESCE(..., 'Unknown')` for categorical fields. The query is identical to the preview above, just without the `LIMIT`.

In [0]:
-- Write the fully cleaned pupils table with audit enrichment
-- COALESCE prefers winning value, falls back to audit value for each field

CREATE OR REPLACE TEMPORARY VIEW pupils_cleaned AS

SELECT p.pupil_id
      ,INITCAP(p.first_name)                                        AS first_name
      ,INITCAP(p.last_name)                                         AS last_name
      ,COALESCE(gl.canonical_value, gl_ape.canonical_value, 'Unknown') AS gender
      ,COALESCE(
        TRY_TO_DATE(p.date_of_birth, 'yyyy-MM-dd'),
        TRY_TO_DATE(p.date_of_birth, 'dd/MM/yyyy'),
        TRY_TO_DATE(p.date_of_birth, 'MM-dd-yyyy')
      )                                                               AS date_of_birth
      ,p.school_urn
      ,COALESCE(ygl.canonical_value, ygl_ape.canonical_value, 'Unknown') AS year_group
      ,COALESCE(el.canonical_value, el_ape.canonical_value, 'Unknown')   AS ethnicity
      ,p.term
      ,p.year
      ,CASE
        WHEN LOWER(TRIM(COALESCE(p.metadata_json:sen_status, ape.sen_status))) = 'ehcp'        THEN 'EHCP'
        WHEN LOWER(TRIM(COALESCE(p.metadata_json:sen_status, ape.sen_status))) = 'sen support' THEN 'SEN Support'
        WHEN LOWER(TRIM(COALESCE(p.metadata_json:sen_status, ape.sen_status))) IN ('none')     THEN 'None'
        ELSE 'Unknown'
      END                                                             AS sen_status
      ,CAST(COALESCE(p.metadata_json:fsm_eligible, ape.fsm_eligible) AS BOOLEAN)   AS fsm_eligible
      ,CAST(COALESCE(p.metadata_json:attendance_pct, ape.attendance_pct) AS DOUBLE) AS attendance_pct
      ,COALESCE(REPLACE(p.metadata_json:address.line1, '"', ''), REPLACE(ape.address_line1, '"', ''))         AS address_line1
      ,COALESCE(REPLACE(p.metadata_json:address.city, '"', ''), REPLACE(ape.address_city, '"', ''))           AS address_city
      ,COALESCE(REPLACE(p.metadata_json:address.postcode, '"', ''), REPLACE(ape.address_postcode, '"', ''))   AS address_postcode
FROM silver.pupils_deduplicated p
LEFT JOIN audit_pupils_enrichment ape
  ON p.pupil_id = ape.pupil_id AND p.term = ape.term AND p.year = ape.year
LEFT JOIN gender_lookup gl
  ON LOWER(TRIM(p.gender)) = gl.raw_value
LEFT JOIN gender_lookup gl_ape
  ON LOWER(TRIM(ape.gender)) = gl_ape.raw_value
LEFT JOIN year_group_lookup ygl
  ON LOWER(TRIM(COALESCE(p.year_group, ''))) = ygl.raw_value
LEFT JOIN year_group_lookup ygl_ape
  ON LOWER(TRIM(COALESCE(ape.year_group, ''))) = ygl_ape.raw_value
LEFT JOIN ethnicity_lookup el
  ON LOWER(TRIM(COALESCE(p.ethnicity, ''))) = el.raw_value
LEFT JOIN ethnicity_lookup el_ape
  ON LOWER(TRIM(COALESCE(ape.ethnicity, ''))) = el_ape.raw_value;

  SELECT * FROM pupils_cleaned LIMIT 12;

### Validating `pupils_cleaned`

We check that:

* The **row count matches the source** (111 rows, 30 distinct pupils)
* Categorical fields (`gender`, `year_group`, `ethnicity`, `sen_status`) have **zero NULLs** — all missing values are now `'Unknown'`
* Non-categorical fields (`date_of_birth`, `fsm_eligible`, `attendance_pct`) may still have NULLs where data is genuinely absent

In [0]:
-- Validate: row count matches source (111), categorical fields have no NULLs
-- Non-categorical fields (dates, booleans, numbers) may still have NULLs

SELECT COUNT(*)              AS total_rows
      ,COUNT(DISTINCT pupil_id) AS distinct_pupils
      ,SUM(CASE WHEN gender = 'Unknown' THEN 1 ELSE 0 END)      AS unknown_gender
      ,COUNT(date_of_birth)   AS non_null_dob
      ,SUM(CASE WHEN year_group = 'Unknown' THEN 1 ELSE 0 END)  AS unknown_year_group
      ,SUM(CASE WHEN ethnicity = 'Unknown' THEN 1 ELSE 0 END)   AS unknown_ethnicity
      ,SUM(CASE WHEN sen_status = 'Unknown' THEN 1 ELSE 0 END)  AS unknown_sen_status
      ,COUNT(fsm_eligible)    AS non_null_fsm
      ,COUNT(attendance_pct)  AS non_null_attendance
FROM silver.pupils_cleaned

---

## Part 3: Extracting Contact Details

Parent/guardian contact information is buried inside `metadata_json` in the pupils data. We extract it into a dedicated `contacts_cleaned` table for two reasons:

1. **Separation of concerns** — contact details are about *people*, not educational data
2. **Completeness** — discarded duplicate rows (in `audit_pupils_duplicate_rows`) may contain contact details not present in the winning rows

We combine contacts from both `pupils_deduplicated` (111 rows) and `audit_pupils_duplicate_rows` (17 rows), then deduplicate to ensure each unique contact record appears only once.

#### Why include `term` and `year`?

Each row in `pupils_cleaned` represents a pupil observation for a specific term and year. Contact details can change over time (e.g. a parent's email or phone number may be updated between terms), so we include `term` and `year` in the contacts table. This ensures joins back to `pupils_cleaned` match the correct observation:

```sql
FROM pupils_cleaned p
JOIN contacts_cleaned c
  ON p.pupil_id = c.pupil_id
  AND p.term = c.term
  AND p.year = c.year
```

Without `term` and `year`, a join on `pupil_id` alone would produce a many-to-many cross-product.

#### Contact JSON structure

```json
{
  "contact": {
    "parent_name": "Sarah Johnson",
    "phone": "07700900001",
    "email": "s.johnson@email.com",
    "address": {
      "line1": "14 Maple Road",
      "city": "London",
      "postcode": "E1 6AN"
    }
  }
}
```

Note that `phone` and `email` are not always present — some contacts only have an address.

In [0]:
-- Preview: extract contact details from BOTH deduplicated and audit tables
-- UNION removes exact duplicates automatically

SELECT pupil_id
      ,REPLACE(metadata_json:contact.parent_name, '"', '')           AS parent_name
      ,REPLACE(metadata_json:contact.phone, '"', '')                 AS phone
      ,REPLACE(metadata_json:contact.email, '"', '')                 AS email
      ,REPLACE(metadata_json:contact.address.line1, '"', '')         AS address_line1
      ,REPLACE(metadata_json:contact.address.city, '"', '')          AS address_city
      ,REPLACE(metadata_json:contact.address.postcode, '"', '')      AS address_postcode
      ,term
      ,year
FROM silver.pupils_deduplicated
WHERE metadata_json:contact IS NOT NULL

UNION

SELECT pupil_id
      ,REPLACE(metadata_json:contact.parent_name, '"', '')           AS parent_name
      ,REPLACE(metadata_json:contact.phone, '"', '')                 AS phone
      ,REPLACE(metadata_json:contact.email, '"', '')                 AS email
      ,REPLACE(metadata_json:contact.address.line1, '"', '')         AS address_line1
      ,REPLACE(metadata_json:contact.address.city, '"', '')          AS address_city
      ,REPLACE(metadata_json:contact.address.postcode, '"', '')      AS address_postcode
      ,term
      ,year
FROM silver.audit_pupils_duplicate_rows
WHERE metadata_json:contact IS NOT NULL

ORDER BY pupil_id, year, term

### Writing `contacts_cleaned`

The write query is similar to the preview, with one addition: `NULLIF` converts empty strings to `NULL` for `phone` and `email` fields. Contact fields are non-categorical, so missing values stay as `NULL` rather than `'Unknown'` — an `'Unknown'` phone number or email address would be misleading.

In [0]:
-- Write the deduplicated contacts table
-- UNION (not UNION ALL) removes exact duplicate rows automatically
-- NULLIF converts empty strings to NULL for consistency

CREATE OR REPLACE TEMPORARY VIEW contacts_cleaned AS

SELECT CONCAT_WS('|', pupil_id, term, year) as pupil_natural_key
      ,pupil_id
      ,term
      ,year
      ,REPLACE(metadata_json:contact.parent_name, '"', '')            AS parent_name
      ,NULLIF(REPLACE(metadata_json:contact.phone, '"', ''), '')      AS phone
      ,NULLIF(REPLACE(metadata_json:contact.email, '"', ''), '')      AS email
      ,REPLACE(metadata_json:contact.address.line1, '"', '')          AS address_line1
      ,REPLACE(metadata_json:contact.address.city, '"', '')           AS address_city
      ,REPLACE(metadata_json:contact.address.postcode, '"', '')       AS address_postcode
FROM silver.pupils_deduplicated
WHERE metadata_json:contact IS NOT NULL

UNION

SELECT CONCAT_WS('|', pupil_id, term, year) as pupil_natural_key
      ,pupil_id
      ,term
      ,year
      ,REPLACE(metadata_json:contact.parent_name, '"', '')            AS parent_name
      ,NULLIF(REPLACE(metadata_json:contact.phone, '"', ''), '')      AS phone
      ,NULLIF(REPLACE(metadata_json:contact.email, '"', ''), '')      AS email
      ,REPLACE(metadata_json:contact.address.line1, '"', '')          AS address_line1
      ,REPLACE(metadata_json:contact.address.city, '"', '')           AS address_city
      ,REPLACE(metadata_json:contact.address.postcode, '"', '')       AS address_postcode
FROM silver.audit_pupils_duplicate_rows
WHERE metadata_json:contact IS NOT NULL;

SELECT * FROM silver.contacts_cleaned LIMIT 12

As with previous notebooks the `schools_cleaned`, `pupils_cleaned` and `contacts_cleaned` are now stored in the following tables:

 - `catalog_40_copper_analysts_training.silver.schools_cleaned`
 - `catalog_40_copper_analysts_training.silver.pupils_cleaned`
 - `catalog_40_copper_analysts_training.silver.contacts_cleaned`

## Final validation

A single query checks row counts and distinct entity counts across all three output tables to confirm everything was written correctly.

In [0]:
-- Final validation: row counts and column types for all three output tables

SELECT 'pupils_cleaned' AS table_name
      ,COUNT(*) AS total_rows
      ,COUNT(DISTINCT pupil_id) AS distinct_entities
FROM silver.pupils_cleaned

UNION ALL

SELECT 'schools_cleaned'
      ,COUNT(*)
      ,COUNT(DISTINCT school_urn)
FROM silver.schools_cleaned

UNION ALL

SELECT 'contacts_cleaned'
      ,COUNT(*)
      ,COUNT(DISTINCT pupil_id)
FROM silver.contacts_cleaned

ORDER BY table_name